# NeuroGolf V12: Zero-Dependency Master Sweep

This notebook contains the entirely self-contained NeuroGolf engine. 
It bypasses all dataset mount issues by embedding the source code as an encoded bundle.

In [ ]:
# 1. Install missing dependencies for ONNX export
!pip install onnxscript onnx --quiet


In [ ]:
import base64
import zipfile
import os
from pathlib import Path

bundle_b64 = "..."

print("🛠️ Extracting NeuroGolf Master Bundle...")
try:
    with open("bundle.zip", "wb") as f:
        f.write(base64.b64decode(bundle_b64))
    
    with zipfile.ZipFile("bundle.zip", "r") as zip_ref:
        zip_ref.extractall(".")
    
    # Validation (Brutal Truth Safeguard)
    expected_files = ["src/neurogolf/constants.py", "scripts/kaggle_solve.py"]
    for f in expected_files:
        if not os.path.exists(f):
            raise RuntimeError(f"❌ Critical file missing after extraction: {f}")
    
    print("✅ Extraction Successful. Library validated.")
    os.remove("bundle.zip")
except Exception as e:
    raise RuntimeError(f"💥 Bundle extraction failed: {str(e)}")

In [ ]:
import sys
sys.path.insert(0, os.path.abspath('src'))
sys.path.insert(0, os.path.abspath('scripts'))

from scripts import kaggle_solve

print("🚀 Starting 400-Task Sweep...")
kaggle_solve.main()